# Careerjet job scraping

In [1]:
import requests, time, random
import pandas as pf
from bs4 import BeautifulSoup

In [ ]:
import requests, time, random
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "it-IT,it;q=0.9,en;q=0.8",
}

def text(el):
    return el.get_text(" ", strip=True) if el else None

def scrape_careerjet(query="data scientist", location="Italia", max_pages=5):
    rows = []
    for page in range(1, max_pages + 1):
        r = requests.get(
            "https://www.careerjet.it/offerte-lavoro",
            params={"s": query, "l": location, "p": page},
            headers=HEADERS, timeout=30,
        )
        if r.status_code != 200:
            print(f"Careerjet page {page}: HTTP {r.status_code}, stopping")
            break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article.job")
        if not cards:
            break
        for c in cards:
            a = c.select_one("h2 a")
            rows.append({
                "source": "careerjet",
                "title": text(a),
                "company": text(c.select_one("p.company")),
                "location": text(c.select_one("ul.location li")),
                "salary": text(c.select_one("ul.salary li")),
                "url": ("https://www.careerjet.it" + a["href"])
                       if a and a.get("href", "").startswith("/") else (a or {}).get("href"),
            })
        time.sleep(random.uniform(2, 4))
    return pd.DataFrame(rows)

In [3]:
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

URL = "https://www.careerjet.it/offerte-lavoro?s=Data+Scientist&l=Italia&p=1"

options = Options()
# Keep Chrome visible: CareerJet may require manual verification.
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

try:
    driver.get(URL)
    print("CareerJet opened in Chrome.")
    print("Complete any verification in Chrome, then return here.")
    input("Press Enter after the job results are visible in Chrome: ")
    WebDriverWait(driver, 30).until(
        lambda browser: browser.find_elements(
            By.CSS_SELECTOR, "article.job, .jobsearch-ResultsList > li, .job"
        )
    )
    soup = BeautifulSoup(driver.page_source, "html.parser")
finally:
    driver.quit()


def get_text(card, selectors):
    for selector in selectors:
        element = card.select_one(selector)
        if element:
            value = element.get_text(" ", strip=True)
            if value:
                return value
    return None


cards = soup.select("article.job, .jobsearch-ResultsList > li, .job")
rows = []
for card in cards:
    title = get_text(card, ["h2 a", "h3 a", ".title a", "a.jobtitle"])
    if title:
        rows.append(
            {
                "job_name": title,
                "company": get_text(card, ["p.company", ".company", ".employer"]),
                "salary": get_text(card, ["ul.salary li", ".salary", ".offer-salary"]),
                "location": get_text(
                    card, ["ul.location li", ".location", ".offer-location"]
                ),
            }
        )

jobs = pd.DataFrame(rows, columns=["job_name", "company", "salary", "location"])
if jobs.empty:
    raise RuntimeError(
        "No jobs were extracted. Complete the verification and confirm that job cards "
        "are visible before pressing Enter."
    )

jobs


CareerJet opened in Chrome.
Complete any verification in Chrome, then return here.


,job_name,company,salary,location
0,Data Scientist - Pharma & Life Sciences,Capgemini,31.270-40.150 € all'anno,Roma
1,Junior Data Scientist,Esprinet Italia,1.000 € al mese,"Vimercate, Monza Brianza"
2,Data Scientist / AI Specialist Categoria prote...,TAS,35.000 € all'anno,Roma
3,Data Scientist – Marketing Mix Modeling,MC Engineering,25.000-40.000 € all'anno,Torino
4,Data Scientist-AI Engineer,Agap2,28.000-43.000 € all'anno,Milano
5,Data Scientist – Marketing Mix Modeling,MC Engineering,25.000-40.000 € all'anno,Torino
6,Data Scientist - Field Marketing,CPM International,None,Milano
7,Data Scientist,R1,None,Roma
8,Stagista Data Scientist,Banco Desio,None,"Desio, Monza Brianza"
9,Data Scientist / Applied Scientist,Workfully,None,Milano
